# Data ingestion
Sessions are identified by `(season, round, session)` — never by event name.

In [4]:
from f1_intelligence.config import load_ingestion_config
from f1_intelligence.ingestion.data_writer import ParquetWriter
from f1_intelligence.ingestion.fastf1_loader import SessionKey, enable_cache, list_sessions, load_session

storage = load_ingestion_config("../configs/ingestion.yaml").storage
enable_cache(storage.cache_uri)
writer = ParquetWriter(storage.raw_uri, storage.compression)

In [5]:
# Which race sessions of 2024 have already happened?
races_2024 = list_sessions(2024, sessions=["R"])
races_2024[:5]

[SessionKey(season=2024, round=1, session='R'),
 SessionKey(season=2024, round=2, session='R'),
 SessionKey(season=2024, round=3, session='R'),
 SessionKey(season=2024, round=4, session='R'),
 SessionKey(season=2024, round=5, session='R')]

In [6]:
# 2024 Spanish GP is round 10
data = load_session(SessionKey(2024, 10, "R"))
data.metadata

core           INFO 	Loading data for Spanish Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:00.015000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '44', '63', '16', '55', '81', '11', '10', '31', '27', '14', '24', '18', '3', '77', '20', '23', '22', '2']


{'season': 2024,
 'round': 10,
 'session': 'R',
 'session_name': 'Race',
 'session_start_utc': '2024-06-23T13:00:00',
 'event_name': 'Spanish Grand Prix',
 'official_event_name': 'FORMULA 1 ARAMCO GRAN PREMIO DE ESPAÑA 2024',
 'country': 'Spain',
 'location': 'Barcelona',
 'event_format': 'conventional',
 'missing_tables': [],
 'fastf1_version': '3.8.3',
 'ingested_at_utc': '2026-10-01T05:08:01.724200+00:00'}

In [7]:
{name: df.shape for name, df in data.tables().items()}

{'laps': (1310, 34),
 'weather': (154, 11),
 'results': (20, 25),
 'track_status': (2, 6),
 'race_control': (105, 12)}

In [8]:
data.laps.head()

,Season,Round,Session,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,2024,10,R,0 days 00:58:20.507000,VER,1,0 days 00:01:23.186000,1.0,1.0,NaT,...,False,Red Bull Racing,0 days 00:56:57.095000,NaT,1,2.0,False,,False,False
1,2024,10,R,0 days 00:59:40.378000,VER,1,0 days 00:01:19.871000,2.0,1.0,NaT,...,False,Red Bull Racing,0 days 00:58:20.507000,NaT,1,2.0,False,,False,True
2,2024,10,R,0 days 01:00:59.742000,VER,1,0 days 00:01:19.364000,3.0,1.0,NaT,...,False,Red Bull Racing,0 days 00:59:40.378000,NaT,1,1.0,False,,False,True
3,2024,10,R,0 days 01:02:20.508000,VER,1,0 days 00:01:20.766000,4.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:00:59.742000,NaT,1,1.0,False,,False,True
4,2024,10,R,0 days 01:03:41.335000,VER,1,0 days 00:01:20.827000,5.0,1.0,NaT,...,False,Red Bull Racing,0 days 01:02:20.508000,NaT,1,1.0,False,,False,True


In [ ]:
# Writes one file per table; returns False if this session is already in the raw layer
writer.write(data, overwrite=storage.overwrite)

In [ ]:
import duckdb

# Every ingested session, queried as one table
duckdb.sql(f"""
    SELECT Season, Round, Session, count(*) AS laps
    FROM read_parquet('{storage.raw_uri}/laps/**/*.parquet', hive_partitioning = true)
    GROUP BY ALL ORDER BY ALL
""").df()